# 🚀 ViSceT5 — **PreSTU SplitOCR + MRA** Pretrain → Finetune · Colab **A100**

Pretrain **PreSTU SplitOCR** (đúng paper): sắp OCR theo thứ tự đọc, cắt ngẫu nhiên → prompt chứa phần đầu, mô hình **sinh phần còn lại từ ảnh** (prefix/target rời nhau, buộc ĐỌC pixel; `full_ocr_prob=0.2` thỉnh thoảng sinh toàn bộ). Không bbox/ground. Kèm **MRA** đúng Feast-Your-Eyes: ViT-base-patch16 **336** (nội suy 2D bicubic từ 224) + ConvNeXt-V2-Base **1024** (đặc trưng CUỐI của CNN bơm vào **3 stage cuối** ViT), VS TẮT, `VISION_LR_SCALE=0.2`.

Chống overfit (dữ liệu ít): **cosine LR + 6 epoch**. Sau pretrain → finetune ViTextVQA giữ MRA.

Nhánh `exp/mra-pretrain`. Runtime → **A100**. Internet ON.

## 1. Clone Codebase & Checkout Nhánh Pretrain
Tự động phát hiện môi trường (Kaggle hoặc Colab), đồng bộ repository từ GitHub và chuyển sang nhánh `exp/pretrain-gen-all` chứa các cải tiến mới nhất.

In [ ]:
import os
import sys

# Tự động phát hiện thư mục làm việc (Kaggle: /kaggle/working | Colab: /content)
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout exp/mra-pretrain
!git pull origin exp/mra-pretrain
!git log --oneline -3

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2. Cài Đặt Môi Trường Chuẩn (Transformers 4.45.2 Cố Định)
Gỡ các phiên bản thư viện mặc định của Kaggle và cài đặt chính xác các phiên bản tương thích từ `requirements.txt`.

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap
!pip install -q --upgrade --no-cache-dir gdown

In [ ]:
# Kiểm tra xác nhận phiên bản môi trường
import torch
import transformers
print(f"✅ PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"✅ Transformers Version: {transformers.__version__} (Yêu cầu cố định: 4.45.2)")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
assert transformers.__version__.startswith("4.45"), f"Cảnh báo: Cần transformers 4.45.x để khớp kiến trúc module, hiện tại là {transformers.__version__}"

In [ ]:
# Giảm OOM do phân mảnh VRAM (Colab) cho MỌI lệnh !python phía dưới (smoke/pretrain/finetune).
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
import torch, gc
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('VRAM alloc conf set; free GPU ready.')

In [ ]:
# 🔑 Tự động nạp HF_TOKEN từ Google Colab Secrets (tab biểu tượng chìa khoá bên trái)
# Script pretrain/finetune sẽ TỰ ĐỘNG đẩy checkpoint mỗi epoch và model tốt nhất lên Hugging Face.
import os
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
        print('✅ Đã kết nối HF_TOKEN từ Google Colab Secrets! Tự động bật auto-push lên Hugging Face.')
    else:
        print("⚠️ Chưa tìm thấy secret 'HF_TOKEN' trong Colab. Hãy bật quyền truy cập ở tab Secrets (🔑) bên trái nếu muốn auto-upload HF.")
except Exception as e:
    print(f'ℹ️ Colab Secrets check: {e}')

# (Tùy chọn) Đặt repo cố định nếu muốn khác username/repo mặc định:
# os.environ['HF_REPO'] = 'username/ViSceT5-mra-pretrain'


## 3. Chuẩn Bị Dữ Liệu Tiền Huấn Luyện (VinText + EVJVQA)
Tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và lưu cache vào `output/pretrain`.

In [ ]:
# Định vị thư mục lưu dataset CSV đồng bộ với visualize và trainer
%env OUTPUT_PATH=./output/pretrain
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

### 3b. Xem thử vài mẫu SplitOCR (kiểm tra prefix/target trước khi train)

In [ ]:
# === Xem thử vài mẫu SplitOCR (prefix vs target) để kiểm tra bằng mắt ===
# VinText: TARGET = GT (labels, sạch), PREFIX = text SwinTextSpotter (silver, nhiễu), box rời nhau.
# EVJVQA: không có GT -> tách trên spotter như bình thường.
import os, random, numpy as np, pandas as pd, torch
from PIL import Image
from data.gt_ocr import load_vintext_gt
from data.collator import (_splitocr_b1_gt, _split_ocr_sequential,
                           _sort_ocr_reading_order, _normalize_text)

OUT = os.environ.get("OUTPUT_PATH", "./output/pretrain")
df = pd.read_csv(os.path.join(OUT, "merged_train.csv"))
print("columns:", list(df.columns))
print("has label_path:", "label_path" in df.columns,
      "| #VinText có GT:", int(df["label_path"].notna().sum()) if "label_path" in df.columns else 0)
random.seed(0)

def show(row, tag):
    ip, op = row["image_path"], row["ocr_path"]
    lp = row["label_path"] if "label_path" in row and pd.notna(row["label_path"]) else None
    W, H = Image.open(ip).convert("RGB").size
    sp = np.load(op, allow_pickle=True).item()
    sp_boxes = torch.tensor(np.asarray(sp.get("boxes")), dtype=torch.float)
    sp_texts = list(sp.get("texts", []))
    if isinstance(lp, str) and os.path.exists(lp):
        gt_t, gt_b = load_vintext_gt(lp, W, H)
        gt_t, gt_b = _sort_ocr_reading_order(gt_t, gt_b)
        pfx, tgt = _splitocr_b1_gt(gt_t, gt_b, sp_boxes, sp_texts, full_ocr_prob=0.2)
        src = "VinText  (PREFIX=spotter SILVER, TARGET=GT sạch)"
    else:
        toks = [_normalize_text(t, lowercase=True).strip() for t in sp_texts
                if isinstance(t, str) and t.strip()]
        bx = sp_boxes[:len(toks)] if sp_boxes.size(0) >= len(toks) else torch.zeros(len(toks), 4)
        toks, bx = _sort_ocr_reading_order(toks, bx)
        pw, _, tw, _ = _split_ocr_sequential(toks, bx, full_ocr_prob=0.2)
        pfx, tgt = " ".join(pw), " ".join(tw)
        src = "EVJVQA   (spotter-only split)"
    print("=" * 78)
    print(f"[{tag}] {os.path.basename(str(ip))} | {src}")
    print("  PROMPT :", ("Generate ocr_text in vi: " + pfx).strip() if pfx else "Generate ocr_text in vi:")
    print("  TARGET :", tgt[:220])

vin = df[df["label_path"].notna()] if "label_path" in df.columns else df.iloc[0:0]
evj = df[df["dataset"].astype(str).str.upper().str.contains("EVJ")] if "dataset" in df.columns else df.iloc[0:0]
for _, r in vin.sample(min(3, len(vin))).iterrows(): show(r, "VinText")
for _, r in evj.sample(min(2, len(evj))).iterrows(): show(r, "EVJVQA")
print("\n✅ Kiểm tra: VinText TARGET phải là GT có dấu/đúng hoa-thường; PREFIX là bản spotter (thường sai/thiếu).")


## 4. Khởi Tạo Trọng Số Mô Hình (ViT5 Base & CLIP ViT)
Khởi tạo `OpenViVQAModel`, tải các trọng số nền tảng ViT5 và CLIP-ViT, kiểm tra tính toàn vẹn số học.

In [ ]:
!python scripts/init_model.py

## 5. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

### Cấu hình tối ưu toàn diện:
* **Epochs:** 10
* **Batch size:** 4 (per device) $\times$ 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** $1\times 10^{-4}$ (ViT5) và $1\times 10^{-5}$ (CLIP ViT unfrozen 4 layers)
* **Loss balance:** $\lambda_{\text{bbox}} = 0.3$
* **Vision Unfreeze:** Top-4 layers (`vision_unfreeze_last_n = 4`) + post-layernorm
* **Target Split:** Spatial Region Clustering (Khoanh vùng cụm không gian)
* **Output dir:** `/kaggle/working/pretrain_output`

### 5a. SMOKE test PreSTU+MRA (vài step) — bắt lỗi wiring trước khi chạy full

In [ ]:
# SMOKE: vài step, bắt lỗi wiring PreSTU SplitOCR + MRA trước khi chạy full.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True VISION_LR_SCALE=0.2 python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --clip_vision_name openai/clip-vit-base-patch16 \
    --clip_image_size 336 \
    --vs_backbone facebook/convnextv2-base-22k-384 \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --pretrain_gen_only True \
    --pretrain_split_mode sequential \
    --pretrain_full_ocr_prob 0.2 \
    --vision_unfreeze_last_n -1 \
    --gradient_checkpointing True \
    --bf16 True \
    --tf32 True \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --smoke_test True \
    --output_dir ./output/pretrain_mra_smoke \
    --logging_dir ./output/pretrain_mra_smoke/logs


### 5b. FULL PreSTU SplitOCR + MRA pretrain (6 epoch)

> ☁️ **Tự động lưu & bảo vệ tiến độ lên Hugging Face Hub:**
> - **Mỗi epoch:** Tự động push `checkpoint-xxx` lên Hugging Face Hub (nếu Colab ngắt giữa chừng, lần chạy sau sẽ TỰ ĐỘNG resume từ checkpoint mới nhất).
> - **Ngay khi pretrain xong:** Tự động lưu và push mô hình tốt nhất (`model.safetensors`, `config.json`, tokenizer) lên repo Hugging Face **NGAY LẬP TỨC** (hoàn toàn độc lập, không cần chờ đến finetune).


In [ ]:
# PreSTU SplitOCR (ĐÚNG paper): sinh phần OCR text còn lại từ ảnh (prefix->target rời nhau).
# split_mode=sequential, full_ocr_prob=0.2 (curriculum độ dài target — hợp dữ liệu ít).
# Chống overfit: cosine LR decay + 6 epoch. MRA: ViT-base-patch16 336 + ConvNeXt-V2-Base 1024 (đặc trưng CUỐI -> 3 stage cuối ViT), VS TẮT.
# Tiết kiệm VRAM chống OOM: gradient_checkpointing True + bf16 True + per_device_train_batch_size 2 x accum 8 (effective batch size 16).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True VISION_LR_SCALE=0.2 python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --clip_vision_name openai/clip-vit-base-patch16 \
    --clip_image_size 336 \
    --vs_backbone facebook/convnextv2-base-22k-384 \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --pretrain_gen_only True \
    --pretrain_split_mode sequential \
    --pretrain_full_ocr_prob 0.2 \
    --vision_unfreeze_last_n -1 \
    --num_train_epochs 6 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.05 \
    --learning_rate 0.0001 \
    --weight_decay 0.01 \
    --gradient_checkpointing True \
    --bf16 True \
    --tf32 True \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --save_total_limit 2 \
    --output_dir ./output/pretrain_mra \
    --logging_dir ./output/pretrain_mra/logs


### 5c. (Tùy chọn) Tải checkpoint pretrain về máy dạng zip

Mô hình pretrain đã được tự động lưu tại `./output/pretrain_mra` và đồng bộ lên Hugging Face Hub. Cell dưới đây cho phép bạn nén và tải trực tiếp file zip bản nhẹ về máy tính cá nhân nếu cần.

In [ ]:
# === Tải checkpoint PRETRAIN về máy (Colab) ===
import os, shutil
CKPT = 'output/pretrain_mra'
assert os.path.isdir(CKPT), f'Không thấy {CKPT} — chạy cell pretrain (5b) trước.'

# Bản NHẸ để warm-start finetune: chỉ model.safetensors + config + tokenizer (bỏ optimizer/rng).
LIGHT = 'pretrain_mra_light'
if os.path.isdir(LIGHT): shutil.rmtree(LIGHT)
os.makedirs(LIGHT, exist_ok=True)
KEEP = ('model.safetensors', 'config.json', 'generation_config.json',
        'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
        'spiece.model', 'added_tokens.json')
for f in KEEP:
    src = os.path.join(CKPT, f)
    if os.path.exists(src): shutil.copy2(src, os.path.join(LIGHT, f))
print('Files (light):', os.listdir(LIGHT))
zip_path = shutil.make_archive('ViSceT5_PreSTU_MRA_pretrain', 'zip', LIGHT)
print('Đã nén:', os.path.abspath(zip_path), '| size:', round(os.path.getsize(zip_path)/1e6, 1), 'MB')
try:
    from google.colab import files
    files.download(zip_path)     # trình duyệt tự tải về
except Exception as e:
    print('Không tự tải (không ở Colab UI?). Tải thủ công ở panel Files bên trái:', zip_path)


## 6. Visualize: model đọc OCR (prompt) vs Ground-Truth (giải phóng GPU sau khi xong)

In [ ]:
# === Visualize PreSTU+MRA: model ĐỌC OCR (prompt) vs GT trên vài ảnh val ===
# Nạp checkpoint pretrain, sinh scene-text từ ẢNH (prompt full-read, không prefix), so với GT.
import os, gc, random, torch, numpy as np, pandas as pd
from PIL import Image
from safetensors.torch import load_file
from configs.model_config import OpenViVQAConfig
from models.openvivqa_model import OpenViVQAModel
from utils.model_utils import safe_load_tokenizer
from data.gt_ocr import load_vintext_gt

CKPT = "output/pretrain_mra"
OUTP = os.environ.get("OUTPUT_PATH", "./output/pretrain")
dev  = "cuda" if torch.cuda.is_available() else "cpu"
if not os.path.exists(os.path.join(CKPT, "model.safetensors")):
    print("Chưa thấy checkpoint ở", CKPT, "— chạy cell pretrain (5b) trước.")
else:
    cfg = OpenViVQAConfig.from_pretrained(CKPT) if os.path.exists(os.path.join(CKPT, "config.json")) else OpenViVQAConfig()
    cfg.pretrain = False
    m = OpenViVQAModel(cfg)
    sd = load_file(os.path.join(CKPT, "model.safetensors"))
    m.load_state_dict({k[7:] if k.startswith("module.") else k: v for k, v in sd.items()}, strict=False)
    m.to(dev).eval()
    tok = safe_load_tokenizer("VietAI/vit5-base", use_fast=False)
    df = pd.read_csv(os.path.join(OUTP, "merged_val.csv"))
    vin = df[df["label_path"].notna()] if "label_path" in df.columns else df
    for _, r in vin.sample(min(4, len(vin)), random_state=0).iterrows():
        img = Image.open(r["image_path"]).convert("RGB"); W, H = img.size
        pv = m.image_processor(images=[img], return_tensors="pt")["pixel_values"].to(dev)
        ids = tok("Generate ocr_text in vi:", return_tensors="pt").input_ids.to(dev)
        with torch.no_grad():
            g = m.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
                           pixel_values=pv, pil_images=[img], max_new_tokens=64, num_beams=1)
        pred = tok.decode(g[0], skip_special_tokens=True)
        gt = " ".join(load_vintext_gt(r["label_path"], W, H)[0]) if pd.notna(r.get("label_path")) else "(EVJVQA no GT)"
        print("=" * 78)
        print("IMG :", os.path.basename(str(r["image_path"])))
        print("GT  :", gt[:180])
        print("PRED:", pred[:180])
    del m, sd; gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print("\n✅ (đã giải phóng GPU) — PRED nên đọc gần đúng scene-text tiếng Việt nếu pretrain học tốt.")


## 7. Chuyển Giao Sang Downstream VQA (Transfer Learning to Fine-Tuning)

Sau khi tiền huấn luyện hoàn tất, mô hình đã sẵn sàng chuyển giao tri thức sang bài toán Scene-Text VQA tiếng Việt (**ViTextVQA**).
Toàn bộ trọng số của `vit5.encoder`, `vit5.decoder`, `qa_clip.vision_model` (4 lớp thích ứng) và `visual_search` được nạp nguyên vẹn vào `training/finetune.py`.

> **Tự động liên kết trọng số Pretrain:**
> - Nếu chạy liền trong cùng session Colab: Finetune tự động nạp từ `./output/pretrain_mra`.
> - Nếu chạy trong session mới (hoặc sau khi reset runtime): Finetune sẽ **TỰ ĐỘNG tải pretrain checkpoint từ repo Hugging Face** của bạn về để warm-start!
> - Khi finetune hoàn tất, mô hình finetune tốt nhất cũng sẽ được tự động push lên Hugging Face Hub.

In [ ]:
# Finetune downstream ViTextVQA, warm-start từ checkpoint PreSTU+MRA, GIỮ MRA (VS off, 768).
# bf16 + TF32 + gradient_checkpointing; batch 2 x accum 4 = effective 8 (chống OOM tuyệt đối).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python training/finetune.py configs/finetune.yaml \
    --dataset_name "ViTextVQA" \
    --clip_vision_name openai/clip-vit-base-patch16 \
    --clip_image_size 336 \
    --vs_backbone facebook/convnextv2-base-22k-384 \
    --model_name_or_path ./output/pretrain_mra \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.00003 \
    --vision_unfreeze_last_n -1 \
    --gradient_checkpointing True \
    --bf16 True \
    --tf32 True \
    --output_dir ./output/finetune_mra \
    --logging_dir ./output/finetune_mra/logs


## 8. Kiểm Tra Mô Hình Đã Lưu Trên Hugging Face Hub

In [ ]:
# === Kiểm tra danh sách checkpoint & model đã được auto-upload trên Hugging Face ===
import os
from huggingface_hub import HfApi
hf_tok = os.environ.get('HF_TOKEN')
if hf_tok:
    api = HfApi(token=hf_tok)
    user = api.whoami().get('name')
    pretrain_repo = os.environ.get('HF_REPO') or f'{user}/ViSceT5-mra-pretrain'
    print(f'🌐 Kiểm tra repo Pretrain: https://huggingface.co/{pretrain_repo}')
    try:
        files = api.list_repo_files(repo_id=pretrain_repo)
        print(f'✅ Repo tồn tại! ({len(files)} tệp):')
        for f in sorted(files)[:15]:
            print(f'  - {f}')
        if len(files) > 15:
            print(f'  ... và {len(files)-15} tệp khác.')
    except Exception as e:
        print(f'ℹ️ Chưa tìm thấy repo {pretrain_repo} (sẽ tự động tạo khi chạy pretrain): {e}')
else:
    print('ℹ️ Chưa có HF_TOKEN. Vui lòng cấp quyền ở tab Secrets (🔑) của Colab.')
